# 3-2절 연습 문제 풀이

이 노트북은 3-2절 연습 문제의 풀이 예시다. 정답이 하나뿐인 문제가 아니므로 다른 구현도 얼마든지 가능하다.

- 본문 예제 코드는 `notebooks/ch03/` 아래 예제 노트북을 참고한다.
- 위에서부터 차례대로 실행한다.

In [ ]:
# 환경 설정 - 공통 라이브러리, 시드 고정, 장치 객체
import sys
sys.path.append('../../')

import random

import numpy as np
import torch
import torch.nn as nn

from code_reference import common
# viz.configure()에서 save_grayscale=True로 지정하면 노트북에 표시되는 시각화 이미지를 파일로 저장함
from code_reference import visualize as viz

viz.configure(save_grayscale=False)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = common.get_device()

# 연습 문제에서 공통으로 사용하는 CSV 로더
# 본문 데이터(ch3_spiral_data.csv)는 열 이름이 x1, x2 이고 정답이 숫자,
# 연습 문제 데이터(ch3_exercise_*.csv)는 열 이름이 x, y 이고 정답이 문자열이다.
import csv

def load_csv(path):
    with open(path, encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    cols = rows[0].keys()
    cx, cy = ('x1', 'x2') if 'x1' in cols else ('x', 'y')
    xs = [[float(r[cx]), float(r[cy])] for r in rows]
    labels = [r['label'] for r in rows]
    classes = sorted(set(labels), key=lambda v: (float(v) if v.replace('.','',1).isdigit() else v))
    idx = {name: i for i, name in enumerate(classes)}
    X = torch.tensor(xs)
    Y = torch.tensor([idx[l] for l in labels])      # 교차 엔트로피용 정수 정답
    return X, Y, classes

## 연습 3-4

[코드 3-4]의 모델 요약 정보 출력에서 Param #은 해당 계층의 파라미터의 수를 나타낸다. 시그모이드 활성화 계층을 포함한 네 개의 계층 객체마다 파라미터의 수가 어떻게 결정되었는지 설명해 보자.

In [ ]:
model = nn.Sequential(
    nn.Linear(2, 4), nn.Sigmoid(),
    nn.Linear(4, 1), nn.Sigmoid(),
)
for i, layer in enumerate(model):
    n_param = sum(p.numel() for p in layer.parameters())
    detail = ''
    if isinstance(layer, nn.Linear):
        detail = (f'가중치 {layer.in_features}x{layer.out_features}='
                  f'{layer.in_features * layer.out_features} + 편향 {layer.out_features}')
    else:
        detail = '학습 파라미터 없음'
    print(f'[{i}] {layer.__class__.__name__:10s} Param # = {n_param:3d}  ({detail})')

- **선형 계층**: 파라미터 수 = (입력 크기 × 출력 크기) + 출력 크기. 앞 층의 모든 뉴런이 다음 층의 모든 뉴런과 연결되고, 뉴런마다 편향이 하나씩 붙는다.
- **시그모이드 활성화 계층**: 정해진 수식으로 값을 변환할 뿐 학습할 값이 없으므로 파라미터가 **0개**다.

## 연습 3-5

선형 계층 객체를 만들 때 bias=False 인자를 지정하면 편향 없이 가중치만 파라미터로 사용하는 선형 계층이 된다. [코드 3-1]에서 정의한 모델 클래스를 수정해 은닉층과 출력층의 편향 파라미터를 제거한 후 모델의 학습 결과를 확인해 보자.

In [ ]:
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
Y = torch.tensor([[0.], [1.], [1.], [0.]])          # XOR

def train_xor(use_bias, epochs=5000, lr=0.5, seed=0):
    torch.manual_seed(seed)
    model = nn.Sequential(
        nn.Linear(2, 4, bias=use_bias), nn.Sigmoid(),
        nn.Linear(4, 1, bias=use_bias), nn.Sigmoid(),
    )
    criterion = nn.BCELoss()
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    for _ in range(epochs):
        loss = criterion(model(X), Y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    pred = (model(X) > 0.5).float()
    return loss.item(), torch.equal(pred, Y), pred.flatten().tolist()

for use_bias in (True, False):
    loss, ok, pred = train_xor(use_bias)
    label = '편향 있음' if use_bias else '편향 없음'
    print(f'{label}: 손실 {loss:.4f}, 예측 {[int(v) for v in pred]}, 정답 여부 {ok}')

편향이 없으면 모든 결정 경계가 **원점을 지나야** 한다는 제약이 생긴다. XOR 데이터는 원점을 지나는 경계들만으로는 나누기 어려워 학습이 잘 되지 않거나 특정 초깃값에서만 성공한다. 편향은 경계를 평행 이동시키는 자유도를 제공한다.

## 연습 3-6

[코드 3-1]에서 정의한 모델 클래스에서 다음과 같이 은닉층의 수와 뉴런의 수를 바꿔 가며 모델의 학습 결과를 확인해 보자.

뉴런의 수가 세 개인 은닉층 두 개를 포함한 다층 퍼셉트론 모델

뉴런의 수가 세 개인 은닉층 여섯 개를 포함한 다층 퍼셉트론 모델

뉴런의 수가 열 개인 은닉층 하나를 가진 다층 퍼셉트론 모델

뉴런의 수가 각각 세 개, 두 개인 은닉층 두 개를 포함한 다층 퍼셉트론 모델

In [ ]:
def build_mlp(hidden_sizes):
    layers = []
    fan_in = 2
    for h in hidden_sizes:
        layers += [nn.Linear(fan_in, h), nn.Sigmoid()]
        fan_in = h
    layers += [nn.Linear(fan_in, 1), nn.Sigmoid()]
    return nn.Sequential(*layers)

configs = {
    '뉴런 3개 은닉층 2개': [3, 3],
    '뉴런 3개 은닉층 6개': [3] * 6,
    '뉴런 10개 은닉층 1개': [10],
}
for name, hidden in configs.items():
    torch.manual_seed(0)
    model = build_mlp(hidden)
    criterion = nn.BCELoss(); optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
    for _ in range(5000):
        loss = criterion(model(X), Y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    ok = torch.equal((model(X) > 0.5).float(), Y)
    n_param = sum(p.numel() for p in model.parameters())
    print(f'{name:22s} 파라미터 {n_param:3d}개, 손실 {loss.item():.4f}, 정답 {ok}')

은닉층을 **깊게** 쌓는 것(3뉴런 6층)이 항상 유리하지는 않다. 시그모이드를 여러 번 통과하면 기울기가 점점 작아져(기울기 소실) 앞쪽 층이 거의 학습되지 않는다. XOR처럼 단순한 문제는 **넓은 은닉층 하나**(10뉴런 1층)로도 충분히 해결된다.

깊은 신경망을 실제로 학습시키는 방법은 8장에서 다룬다.

## 연습 3-7

XNOR 게이트는 XOR 게이트와 상반된 결과를 출력한다. 두 게이트를 동시에 나타낸 연산표는 다음과 같다.

표 3-2 XOR 게이트와 XNOR 게이트의 연산표

| 입력 1 | 입력 2 | XOR 게이트 출력 | XNOR 게이트 출력 |
|---|---|---|---|
| 0 | 0 | 0 | 1 |
| 0 | 1 | 1 | 0 |
| 1 | 0 | 1 | 0 |
| 1 | 1 | 0 | 1 |

XOR 게이트와 XNOR 게이트의 결과를 동시에 예측하는 다층 퍼셉트론 모델을 만들어 보자.

In [ ]:
# XNOR = XOR의 반대
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
Y_xnor = torch.tensor([[1.], [0.], [0.], [1.]])

torch.manual_seed(0)
model = nn.Sequential(nn.Linear(2, 4), nn.Sigmoid(), nn.Linear(4, 1), nn.Sigmoid())
criterion = nn.BCELoss(); optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
for _ in range(5000):
    loss = criterion(model(X), Y_xnor)
    optimizer.zero_grad(); loss.backward(); optimizer.step()

pred = (model(X) > 0.5).float()
print(f'{"입력":>10} {"XNOR 예측":>10} {"정답":>6}')
for x, p, y in zip(X.tolist(), pred.flatten().tolist(), Y_xnor.flatten().tolist()):
    print(f'{str([int(v) for v in x]):>10} {int(p):>10} {int(y):>6}')
print(f'\n정답 여부: {torch.equal(pred, Y_xnor)}')

XNOR도 XOR와 마찬가지로 선형 분리가 불가능하므로 은닉층이 필요하다. 정답 텐서만 XOR의 반대로 바꾸면 같은 구조로 학습된다.

## 연습 3-8

깃허브 저장소의 data 디렉터리에 있는 ch3_exercise_1.csv 파일에는 [그림 3-2]의 왼쪽 그림과 같은 분포의 데이터가, ch3_exercise_2.csv 파일에는 오른쪽 그림과 같은 분포의 데이터가 저장되어 있다. 두 파일의 데이터를 각각 분류할 수 있는 모델을 만들어 보자.

제공되는 데이터는 CSV 형식의 텍스트 파일로, x 좌표의 값(x), y 좌표의 값(y), 정답(label)을 쉼표로 구분해 한 행에 샘플 하나씩 기록되어 있다. 단 첫 번째 행은 데이터 샘플이 아니라 열의 이름이다. 참고로 분포가 같을 뿐 [그림 3-2]를 그릴 때 사용한 데이터가 아니므로, 전체 샘플의 수와 시각화 그래프의 모습은 [그림 3-2]와 다르다.

힌트: 이 파일의 정답은 숫자가 아니라 '바깥 원', '안쪽 원'과 같은 문자열이다. 본문에서 설명한 것처럼 클래스마다 인덱스를 부여해 변환한 후 사용해야 한다.

In [ ]:
def train_binary(path, hidden=8, epochs=3000, lr=0.1):
    X, Y, classes = load_csv(path)
    Y = Y.float().unsqueeze(1)              # BCELoss는 (N, 1) 실수형 정답을 받는다.
    torch.manual_seed(SEED)
    model = nn.Sequential(nn.Linear(2, hidden), nn.ReLU(),
                          nn.Linear(hidden, 1), nn.Sigmoid())
    criterion = nn.BCELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    for _ in range(epochs):
        loss = criterion(model(X), Y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    acc = (((model(X) > 0.5).float() == Y).float().mean() * 100).item()
    print(f'{path.split("/")[-1]}: 클래스 {classes}, 정확도 {acc:.2f}%')
    return model, X, Y

for name in ['ch3_exercise_1.csv', 'ch3_exercise_2.csv']:
    train_binary(f'../../data/{name}')

CSV의 정답은 '바깥 원'·'안쪽 원' 같은 문자열이므로, 정렬한 클래스 이름에 0부터 인덱스를 매겨 숫자로 바꿔 사용한다.

두 데이터 모두 직선 하나로는 나눌 수 없는 분포(동심원, 곡선 경계)라 은닉층이 필요하다.

## 연습 3-9

[도전 문제] 본문의 은닉층 역할 설명을 바탕으로, 두 개의 은닉층을 포함한 다층 퍼셉트론에서 첫 번째와 두 번째 은닉층의 뉴런의 수가 각각 세 개, 두 개인 모델과 두 개, 세 개인 모델의 차이를 유추해 보자.

힌트: 각 은닉층이 만드는 결정 경계의 수와 은닉 공간의 차원 수에 주목해 보자.

### 풀이

**은닉층 뉴런 수 3 → 2 인 모델**
- 첫 번째 은닉층이 결정 경계 **세 개**를 만들어 입력 공간을 여러 조각으로 나눈다.
- 두 번째 은닉층은 그 세 조각의 조합을 **2차원 은닉 공간**으로 압축한다.
- 먼저 넓게 나눈 뒤 요약하는 구조라, 복잡한 경계를 만든 다음 정리하는 흐름이다.

**은닉층 뉴런 수 2 → 3 인 모델**
- 첫 번째 은닉층이 결정 경계 **두 개**만 만들어 입력 공간을 최대 네 조각으로만 나눈다.
- 이 시점에서 정보가 2차원으로 좁혀지므로, 두 번째 층에서 뉴런을 세 개로 늘려도 **이미 잃은 정보는 복원되지 않는다**.

정리하면 **앞쪽 층의 뉴런 수가 표현력의 상한**을 정한다. 일반적으로 입력에 가까운 층을 넓게 두고 뒤로 갈수록 좁히는 구성이 자주 쓰이는 이유다.

In [ ]:
# 두 구성을 같은 데이터로 비교해 본다.
X, Y, classes = load_csv('../../data/ch3_exercise_1.csv')
Y = Y.float().unsqueeze(1)
for hidden in ([3, 2], [2, 3]):
    torch.manual_seed(SEED)
    model = nn.Sequential(nn.Linear(2, hidden[0]), nn.ReLU(),
                          nn.Linear(hidden[0], hidden[1]), nn.ReLU(),
                          nn.Linear(hidden[1], 1), nn.Sigmoid())
    criterion = nn.BCELoss(); optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
    for _ in range(3000):
        loss = criterion(model(X), Y)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    acc = (((model(X) > 0.5).float() == Y).float().mean() * 100).item()
    print(f'은닉층 {hidden[0]} -> {hidden[1]}: 정확도 {acc:.2f}%')